In [1]:
import os
from mcp import ClientSession, StdioServerParameters
from contextlib import AsyncExitStack
from mcp.client.stdio import stdio_client
from typing import Optional
import asyncio
import sys
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
ANTHROPIC_API_KEY = os.getenv("CLAUDE_API_KEY")

In [38]:
class MCPClient:
    def __init__(self) -> None:
        self.anthropic = Anthropic(api_key=ANTHROPIC_API_KEY)
        self.exit_stack = AsyncExitStack()
        self.session: Optional[ClientSession] = None
        self.tools = []
        self.messages = []

    async def connect_to_server(self, path_to_server_script: str):
        """
        Connect to MCP server

        Args:
            path_to_server_script: path to MCP server file (.py or .js)
        """
        print(path_to_server_script)
        is_python = path_to_server_script.endswith(".py")
        is_js = path_to_server_script.endswith(".js")

        if not is_python and not is_js:
            raise ValueError("Server script must be .py or .js")

        command = "python" if is_python else "node"
        server_params = StdioServerParameters(
            command=command,
            args=[path_to_server_script],
            env=None,
        )

        stdio_transport =  await self.exit_stack.enter_async_context(stdio_client(server_params))
        self.stdio, self.write = stdio_transport
        self.session = await self.exit_stack.enter_async_context(ClientSession(self.stdio, self.write))

        await self.session.initialize()

        response = await self.session.list_tools()
        tools = response.tools
        print(f"Connected to server with tools: {[tool.name for tool in tools]}") 

    # async def process_query(self, query: str):
    #     try:
    #         messages = [{
    #             "role": "user",
    #             "content": query
    #         }]

    #         response = await self.session.list_tools()
    #         tools = response.tools
    #         available_tools = [{
    #             "name": tool.name,
    #             "description": tool.description,
    #             "input_schema": tool.inputSchema
    #         } for tool in tools]
    #         print(available_tools)

    #         response = self.anthropic.messages.create(
    #             max_tokens=1024,
    #             messages=messages,
    #             model="claude-sonnet-4-20250514",
    #             tools=available_tools
    #         )

    #         final_texts = []
    #         assistant_messages = []
    #         print(f'final texts: {final_texts}')
    #         print(f"assistant message: {assistant_messages}")

    #         for resp in response.content:
    #             if resp.type == "text":
    #                 final_texts.append(resp.text)
    #                 assistant_messages.append(resp)
    #             elif resp.type == "tool_use":
    #                 assistant_messages.append(resp)
    #                 print(f"assistant message: {assistant_messages}")

    #                 tool_params, tool_name = resp.input, resp.name
    #                 print(tool_name, tool_params)
    #                 text_for_tool = f"Calling tool {tool_name} with arguments {tool_params}"
    #                 final_texts.append(text_for_tool)
    #                 print(f'final texts: {final_texts}')

    #                 messages.append({
    #                     'role': 'assistant',
    #                     'content': assistant_messages
    #                 })
                    
    #                 tool_response = await self.session.call_tool(tool_name, tool_params)
    #                 messages.append({
    #                     'role': 'user',
    #                     'content': [
    #                         {
    #                             'type': 'tool_result',
    #                             'tool_use_id': resp.id,
    #                             'content': tool_response.content
    #                         }
    #                     ]
    #                 })
    #                 print(f'messages: {messages}')

    #                 llm_response = self.anthropic.messages.create(
    #                     max_tokens=256,
    #                     messages=messages,
    #                     model="claude-sonnet-4-20250514",
    #                     tools=available_tools
    #                 )

    #                 print(llm_response.content)

    #         return "\n".join(final_texts)

    async def process_query(self, query: str) -> str:
        """Process a query using Claude and available tools"""
        messages = [
            {
                "role": "user",
                "content": query
            }
        ]
        print(f'messages: {messages}')

        response = await self.session.list_tools()
        available_tools = [{
            "name": tool.name,
            "description": tool.description,
            "input_schema": tool.inputSchema
        } for tool in response.tools]

        # Initial Claude API call
        response = self.anthropic.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1000,
            messages=messages,
            tools=available_tools
        )

        # Process response and handle tool calls
        final_text = []
        assistant_message_content = []
        print(f'final texts: {final_text}')
        print(f"assistant message: {assistant_message_content}")

        for content in response.content:
            if content.type == 'text':
                final_text.append(content.text)
                assistant_message_content.append(content)
            elif content.type == 'tool_use':
                tool_name = content.name
                tool_args = content.input

                # Execute tool call
                result = await self.session.call_tool(tool_name, tool_args)
                final_text.append(f"[Calling tool {tool_name} with args {tool_args}]")

                assistant_message_content.append(content)
                messages.append({
                    "role": "assistant",
                    "content": assistant_message_content
                })
                messages.append({
                    "role": "user",
                    "content": [
                        {
                            "type": "tool_result",
                            "tool_use_id": content.id,
                            "content": result.content
                        }
                    ]
                })
                print(f'final texts: {final_text}')
                print(f"assistant message: {assistant_message_content}")
                print(f'messages: {messages}')

                # Get next response from Claude
                response = self.anthropic.messages.create(
                    model="claude-sonnet-4-20250514",
                    max_tokens=1000,
                    messages=messages,
                    tools=available_tools
                )

                final_text.append(response.content[0].text)

        return "\n".join(final_text)
        # except Exception as e:
        #     print(f"error: {e}")

In [39]:
async def main():
    client = MCPClient()
    try:
        await client.connect_to_server(path_to_server_script="../weather-mcp/main.py")
        response = await client.process_query(query="Give me weather and weather comments of Mumbai")
        # print(response.content.text)
        # print(response.content.type)

        return response
    except Exception as e:
        print(f"Error: {e}")

res = await main()
res

../weather-mcp/main.py
Connected to server with tools: ['get_alerts', 'get_weather']
messages: [{'role': 'user', 'content': 'Give me weather and weather comments of Mumbai'}]
final texts: []
assistant message: []
final texts: ["I'll get the current weather information and any weather alerts for Mumbai.", "[Calling tool get_weather with args {'city': 'Mumbai', 'variable': 'temp'}]"]
assistant message: [TextBlock(citations=None, text="I'll get the current weather information and any weather alerts for Mumbai.", type='text'), ToolUseBlock(id='toolu_01FfB8TPw686YnLBtPoKoBn6', caller=DirectCaller(type='direct'), input={'city': 'Mumbai', 'variable': 'temp'}, name='get_weather', type='tool_use')]
messages: [{'role': 'user', 'content': 'Give me weather and weather comments of Mumbai'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text="I'll get the current weather information and any weather alerts for Mumbai.", type='text'), ToolUseBlock(id='toolu_01FfB8TPw686YnLBtPoKoBn6', cal

In [19]:
async def main():
    client = MCPClient()
    try:
        await client.connect_to_server(path_to_server_script="../weather-mcp/main.py")
        response = await client.process_query(query="Explain different components of langchain")
        # print(response.content.text)
        # print(response.content.type)

        return response
    except Exception as e:
        print(f"Error: {e}")

In [20]:
res = await main()
res

../weather-mcp/main.py
Connected to server with tools: ['get_alerts', 'get_weather']
[{'name': 'get_alerts', 'description': '\nFor getting weather alert for a city using the OpenWeather API. \nParams:\n    - city: city for which we want to get temperature\n', 'input_schema': {'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_alertsArguments', 'type': 'object'}}, {'name': 'get_weather', 'description': "\nFor getting weather (a particular variable - like temperature, wind speed, etc) for a city using the OpenWeather API. \nParams:\n    - city: city for which we want to get temperature\n    - variable: variable like 'speed' (for wind speed), 'deg' (for wind direction in degrees), 'temp' (for temperature), 'feels_like' (for temperature feels like), 'temp_min' (for minimum temperature), 'temp_max' (for maximum temperature), 'pressure' (for pressure), 'humidity' (for humidity), 'sea_level' (for sea level), 'grnd_level' (for ground level)\n", 'in

'LangChain is a comprehensive framework designed for building applications with large language models (LLMs). Here are its key components:\n\n## Core Components\n\n### 1. **LLMs and Chat Models**\n- **LLMs**: Interface for text completion models (like GPT-3)\n- **Chat Models**: Interface for conversational models (like ChatGPT)\n- Provides unified API for different model providers (OpenAI, Anthropic, Hugging Face, etc.)\n\n### 2. **Prompts**\n- **PromptTemplates**: Create dynamic prompts with variables\n- **ChatPromptTemplate**: For structured chat conversations\n- **FewShotPromptTemplate**: Include examples in prompts\n- **Pipeline prompts**: Chain multiple prompt templates\n\n### 3. **Chains**\n- **LLMChain**: Basic chain combining LLM + prompt\n- **Sequential chains**: Run chains in sequence\n- **Router chains**: Route inputs to different sub-chains\n- **Transform chains**: Modify data between chain steps\n- **Custom chains**: Build specialized workflows\n\n### 4. **Memory**\n- **Co

In [21]:
res

'LangChain is a comprehensive framework designed for building applications with large language models (LLMs). Here are its key components:\n\n## Core Components\n\n### 1. **LLMs and Chat Models**\n- **LLMs**: Interface for text completion models (like GPT-3)\n- **Chat Models**: Interface for conversational models (like ChatGPT)\n- Provides unified API for different model providers (OpenAI, Anthropic, Hugging Face, etc.)\n\n### 2. **Prompts**\n- **PromptTemplates**: Create dynamic prompts with variables\n- **ChatPromptTemplate**: For structured chat conversations\n- **FewShotPromptTemplate**: Include examples in prompts\n- **Pipeline prompts**: Chain multiple prompt templates\n\n### 3. **Chains**\n- **LLMChain**: Basic chain combining LLM + prompt\n- **Sequential chains**: Run chains in sequence\n- **Router chains**: Route inputs to different sub-chains\n- **Transform chains**: Modify data between chain steps\n- **Custom chains**: Build specialized workflows\n\n### 4. **Memory**\n- **Co

In [ ]:
print(res.content)

[TextBlock(citations=None, text="I'll fetch multiple weather details for Mumbai simultaneously! Let me make all the requests at once.", type='text'), ToolUseBlock(id='toolu_015doUGTjA3xGrhfWtJhCX7m', caller=DirectCaller(type='direct'), input={'city': 'Mumbai', 'variable': 'temp'}, name='get_weather', type='tool_use'), ToolUseBlock(id='toolu_01X1fZ7B38tsG2qk5b8Fr5pQ', caller=DirectCaller(type='direct'), input={'city': 'Mumbai', 'variable': 'feels_like'}, name='get_weather', type='tool_use'), ToolUseBlock(id='toolu_01EMKZ4Yv594QtRfy6hPrcMX', caller=DirectCaller(type='direct'), input={'city': 'Mumbai', 'variable': 'temp_min'}, name='get_weather', type='tool_use'), ToolUseBlock(id='toolu_01JEbpUQ641DrzXriHr9Eop9', caller=DirectCaller(type='direct'), input={'city': 'Mumbai', 'variable': 'temp_max'}, name='get_weather', type='tool_use'), ToolUseBlock(id='toolu_013fApQHo3UCLfnCGNdqUwrQ', caller=DirectCaller(type='direct'), input={'city': 'Mumbai', 'variable': 'humidity'}, name='get_weather', 

In [7]:
for i in range(len(res.content)):
    print(res.content[i])

AttributeError: 'NoneType' object has no attribute 'content'

In [64]:
res.content[2].input, res.content[2].name

({'city': 'Mumbai', 'variable': 'feels_like'}, 'get_weather')

In [ ]:
async def main():
    client = MCPClient()
    try:
        await client.connect_to_server(path_to_server_script="../weather-mcp/main.py")
        response = await client.process_query(query="Give me weather and weather comments of Mumbai")
        inp, name = response.content[2].input, response.content[2].name

        updated_message = []
        for resp in response.content:
            if resp.type == "text":
                pass
            elif resp.type == "tool_use":
                tool_params, tool_name = resp.input, resp.name
                print(tool_name, tool_params)
                text_for_tool = f"Calling tool {tool_name} with arguments {tool_params}"
                tool_response = await client.session.call_tool(tool_name, tool_params)
                tool_answer = tool_response.content[0]['text']

                client.anthropic.messages.create(
                    max_tokens=256,
                    messages=,
                    model=
                )

                llm_response = client


        return response
    except Exception as e:
        print(f"Error: {e}")

res = await main()
res

../weather-mcp/main.py
Connected to server with tools: ['get_alerts', 'get_weather']
[{'name': 'get_alerts', 'description': '\nFor getting weather alert for a city using the OpenWeather API. \nParams:\n    - city: city for which we want to get temperature\n', 'input_schema': {'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_alertsArguments', 'type': 'object'}}, {'name': 'get_weather', 'description': "\nFor getting weather (a particular variable - like temperature, wind speed, etc) for a city using the OpenWeather API. \nParams:\n    - city: city for which we want to get temperature\n    - variable: variable like 'speed' (for wind speed), 'deg' (for wind direction in degrees), 'temp' (for temperature), 'feels_like' (for temperature feels like), 'temp_min' (for minimum temperature), 'temp_max' (for maximum temperature), 'pressure' (for pressure), 'humidity' (for humidity), 'sea_level' (for sea level), 'grnd_level' (for ground level)\n", 'in

Message(id='msg_01RsxM2XN1caEHuZsDVgEDYY', container=None, content=[TextBlock(citations=None, text="I'll fetch the weather details for Mumbai simultaneously! Let me make all the necessary calls at once.", type='text'), ToolUseBlock(id='toolu_01Wpa2Hpkp9Ux4Pi6tTdtosx', caller=DirectCaller(type='direct'), input={'city': 'Mumbai', 'variable': 'temp'}, name='get_weather', type='tool_use'), ToolUseBlock(id='toolu_01Xvwdh3UPieG1XECSKtnL1Q', caller=DirectCaller(type='direct'), input={'city': 'Mumbai', 'variable': 'feels_like'}, name='get_weather', type='tool_use'), ToolUseBlock(id='toolu_01DXnNUELTq962Tq1yKBzr3x', caller=DirectCaller(type='direct'), input={'city': 'Mumbai', 'variable': 'temp_min'}, name='get_weather', type='tool_use'), ToolUseBlock(id='toolu_01Rh9NVoYR3AsyYo8cgrbzBL', caller=DirectCaller(type='direct'), input={'city': 'Mumbai', 'variable': 'temp_max'}, name='get_weather', type='tool_use'), ToolUseBlock(id='toolu_019W8JwL2JjUdPCbRDECfUBe', caller=DirectCaller(type='direct'), i

In [ ]:
x = []
x.append(['asfg'])
x

In [78]:
# Using both filesystem and database MCP to get and compare forecast values with real sales values
# AI assistant will read forecasted.txt doc, read sales db data and compare results

# We'll use filesystem-MCP for reading files, database-MCP for reading database and LLM for comparison

import asyncio
from contextlib import AsyncExitStack
from typing import Optional
from anthropic import Anthropic
from mcp.client.stdio import stdio_client
from mcp import ClientSession, StdioServerParameters
import os
from dotenv import load_dotenv

load_dotenv()
ANTHROPIC_API_KEY = os.getenv("CLAUDE_API_KEY")

filesystem_server = StdioServerParameters(
    command="python",
    args=["../filesystem-mcp/main.py"],
    env=None
)

database_server = StdioServerParameters(
    command="python",
    args=["../database-mcp/main.py"],
    env=None
)

class MCPClient:
    def __init__(self) -> None:
        self.anthropic = Anthropic(api_key=ANTHROPIC_API_KEY)
        self.exit_stack = AsyncExitStack()
        self.session: Optional[ClientSession] = None

    async def connect_to_servers(self):
        try:
            fs_transport = await self.exit_stack.enter_async_context(stdio_client(filesystem_server))
            db_transport = await self.exit_stack.enter_async_context(stdio_client(database_server))

            fs_read, fs_write = fs_transport
            db_read, db_write = db_transport
        
            self.fs_session = await self.exit_stack.enter_async_context(ClientSession(fs_read, fs_write))
            self.db_session = await self.exit_stack.enter_async_context(ClientSession(db_read, db_write))

            await self.fs_session.initialize()
            await self.db_session.initialize()

            fs_tools_list = await self.fs_session.list_tools()
            db_tools_list = await self.db_session.list_tools()

            fs_tools = fs_tools_list.tools
            db_tools = db_tools_list.tools

            print(f"Connected to Filesystem and Database servers with tools: {fs_tools}, {db_tools}")
        except Exception as e:
            print(f"error: {e}")

    async def clean_sessions(self):
        await self.exit_stack.aclose()

    async def process_query(self, query):

        messages = [{
            'role': 'user',
            'content': query
        }]

        fs_tools_list = await self.fs_session.list_tools()
        db_tools_list = await self.db_session.list_tools()

        fs_tools = fs_tools_list.tools
        db_tools = db_tools_list.tools

        available_tools = [{
            'name': f"fs_{tool.name}",
            'description': tool.description,
            'input_schema': tool.inputSchema
        } for tool in fs_tools] + [{
            'name': f"db_{tool.name}",
            'description': tool.description,
            'input_schema': tool.inputSchema
        } for tool in db_tools]

        llm_parsing = self.anthropic.messages.create(
            max_tokens=512,
            messages=messages,
            model="claude-sonnet-4-20250514",
            tools=available_tools
        )

        steps = llm_parsing.content
        print(steps)

        result_steps = []
        # llm_context = []

        for step in steps:
            print(step)
            if step.type == "text":
                print('text')
                result_steps.append(step.text)
                # llm_context.append(step)
            elif step.type == "tool_use":
                print('tool-use')
                tool_name, tool_params = step.name, step.input
                result_steps.append(f"Using tool {tool_name} with args {tool_params}")
                print(result_steps)
                if tool_name.startswith('fs'):
                    tool_name = tool_name[3:]
                    tool_result = await self.fs_session.call_tool(name=tool_name, arguments=tool_params)
                elif tool_name.startswith('db'):
                    tool_name = tool_name[3:]
                    tool_result = await self.db_session.call_tool(name=tool_name, arguments=tool_params)
                print(tool_result)
                # llm_context.append(step)
                messages.append({
                    'role': 'assistant',
                    'content': [step]
                })
                messages.append({
                    'role': 'user',
                    'content': [{
                        'type': "tool_result",
                        'tool_use_id': step.id,
                        'content': tool_result.content
                    }]
                })
        print(messages)
        response = self.anthropic.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            messages=messages
        )
        print(response.content)
        result_steps.append(response.content[0].text)

        return '\n'.join(result_steps)

async def main():
    client = MCPClient()
    try:
        await client.connect_to_servers()
        response = await client.process_query(query="Check if the actual sales for the month of april matches forecast for april. forecast is present in forecast.txt file in data folder.")
        # print(response.content.text)
        # print(response.content.type)

        return response
    except Exception as e:
        print(f"Error: {e}")
    finally:
        await client.clean_sessions()

res = await main()
res


Connected to Filesystem and Database servers with tools: [Tool(name='get_files_list', title=None, description='get list of all the files inside folder folder_name', inputSchema={'properties': {'folder_name': {'title': 'Folder Name', 'type': 'string'}}, 'required': ['folder_name'], 'title': 'get_files_listArguments', 'type': 'object'}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None), Tool(name='read_file', title=None, description='read file file_name inside folder folder_name', inputSchema={'properties': {'file_name': {'title': 'File Name', 'type': 'string'}, 'folder_name': {'title': 'Folder Name', 'type': 'string'}}, 'required': ['file_name', 'folder_name'], 'title': 'read_fileArguments', 'type': 'object'}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None), Tool(name='search_file', title=None, description='search file file_name inside folder folder_name', inputSchema={'properties': {'file_name': {'title': 'File Name', 'type': 'strin

"I'll help you check if the actual sales for April match the forecast. Let me first get the actual sales data for April and then read the forecast file.\nUsing tool db_get_sales with args {'month': 'april'}\nUsing tool fs_read_file with args {'file_name': 'forecast.txt', 'folder_name': 'data'}\nBased on the data I retrieved:\n\n- **Actual sales for April**: $9,700\n- **Forecast for April**: $9,500\n\nThe actual sales for April ($9,700) were **$200 higher** than the forecast ($9,500). This represents a positive variance of about 2.1% above the forecast.\n\nSo while the actual sales don't exactly match the forecast, they exceeded expectations by $200, which is generally a positive outcome from a business perspective."

In [80]:
print(res)

I'll help you check if the actual sales for April match the forecast. Let me first get the actual sales data for April and then read the forecast file.
Using tool db_get_sales with args {'month': 'april'}
Using tool fs_read_file with args {'file_name': 'forecast.txt', 'folder_name': 'data'}
Based on the data I retrieved:

- **Actual sales for April**: $9,700
- **Forecast for April**: $9,500

The actual sales for April ($9,700) were **$200 higher** than the forecast ($9,500). This represents a positive variance of about 2.1% above the forecast.

So while the actual sales don't exactly match the forecast, they exceeded expectations by $200, which is generally a positive outcome from a business perspective.
